# Tools

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

Use a decorator to transfer a python function to LLM tool

In [ ]:
from langchain_core.tools import tool
 
@tool   # This funciton will be used as a tool
def test(query: str) -> str:
    """Test function"""
    return "42f"
 
print(f"test.name:{test.name}")
print(f"test.description:{test.description}")
print(f"test.args:{test.args}")

test.name:test
test.description:Test function
test.args:{'query': {'title': 'Query', 'type': 'string'}}


In [3]:
test.run("a")

'42f'

In [5]:
from pydantic import BaseModel, Field
class SearchInput(BaseModel):
    query: str = Field(description="Thing to search for")

@tool(args_schema=SearchInput)  
def test(query: str) -> str:
    """Test function"""
    return "42f"

print(f"test.args:{test.args}")

test.args:{'query': {'description': 'Thing to search for', 'title': 'Query', 'type': 'string'}}


## Weather request

In [6]:
import requests

@tool
def getweather(location: str) -> str:
    '''
    Fetch current weather for given location
    Args: location(str)
    Returns: str: Text about weather condition
    '''
    url="https://uapis.cn/api/v1/misc/weather"
    try:
        response=requests.get(url, params={"city":location}, timeout=5)
        response.raise_for_status()
        data=response.json()
        city_name=data.get("city", location)
        weather=data.get("weather", "")
        temperature=data.get("temperature", "")
        wind_direction=data.get("wind_direction", "")
        wind_power=data.get("wind_power", "")
        humidity=data.get("humidity", "")
        report_time=data.get("report_time", "")
        
        result = (f"{city_name} current weather: {weather}, {temperature}C\n"
          f"{wind_direction}{wind_power}, humidity {humidity}% ({report_time})")
        return result
    except requests.exceptions.Timeout:
        return "TimeoutError"
    except Exception as e:
        return f"error {e}"

In [7]:
getweather.run("Shanghai")

'上海 current weather: 晴, 30C\n东北风2级, humidity 71% (9 分钟前发布)'

### Langgraph

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import create_agent

agent=create_agent(model="deepseek-chat", tools=[getweather])

#   Streaming Message Chunks
inputs = {"messages": [HumanMessage(content="What's the weather like in Shanghai")]}

for token, output in agent.stream(inputs, stream_mode="messages"):  # Only message
    print(token.content, end="", flush=True)    #   Output token by token
   

Let me check the current weather in Shanghai for you.上海 current weather: 晴, 30C
西北风2级, humidity 73% (9 分钟前发布)The current weather in Shanghai is:

🌤 **Sunny (晴)**
- **Temperature**: 30°C
- **Wind**: Northwest wind, light breeze (level 2)
- **Humidity**: 73%

It's a warm and sunny day in Shanghai! ☀️

The graph structure is a ring. After the data enters the tool node (getweather) and finishes execution, it will loop back to the large language model node. The model receives the weather string and proceeds to the second round of reasoning.

### Langchain

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="deepseek-chat", temperature=0,
                  api_key=os.getenv("DEEPSEEK_API_KEY"),
                  base_url=os.getenv("DEEPSEEK_BASE_URL"))

model_with_tool = model.bind_tools([getweather], tool_choice="getweather")
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful weather assistant."),
    ("human", "{input}")
])


In [ ]:
weather_chain=prompt|model_with_tool
result = weather_chain.invoke({"input": "What's the weather like in Shanghai?"})
print(result)   # Return AImessage with tool calls, but did not run the tool

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 308, 'total_tokens': 345, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 308}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'a6587563-9e8b-42f7-b9fc-d1bb060ed33b', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f4cd9-d767-72c2-a46f-c1e32fd892ed-0' tool_calls=[{'name': 'getweather', 'args': {'location': 'Shanghai'}, 'id': 'call_00_z2kNAkgZRDEtHbnqHg1M7390', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 308, 'output_tokens': 37, 'total_tokens': 345, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


In [12]:
def call_weather_tool(ai_message):
    tool_call = ai_message.tool_calls[0] 
    args = tool_call["args"] 
    return getweather.invoke(args)

weather_chain=prompt|model_with_tool|RunnableLambda(call_weather_tool)  # Run getweather function
result = weather_chain.invoke({"input": "What's the weather like in Shanghai?"})
print(result) 

上海 current weather: 多云, 28C
东南风2级, humidity 83% (9 分钟前发布)


## Multiple Functions

In [17]:
@tool
def getvideo(bvid: str) -> dict:
    '''
    Obtain bid and relevant information by given aid
    Args: aid(str)
    Return: info(dict)
    '''
    url="https://uapis.cn/api/v1/social/bilibili/videoinfo"
    try:
        response=requests.get(url, params={"bvid": bvid}, timeout=5)
        response.raise_for_status()
        data=response.json()

        bvid=data.get("bvid", "")
        title=data.get("title", "")
        desc=data.get("desc", "")
        owner=data.get("owner", "")
        stat=data.get("stat", "")

        result = {
            "bvid": bvid,
            "title": title,
            "desc": desc,
            "owner": owner, 
            "stat": stat 
            }
        return result
    
    except requests.exceptions.Timeout:
        return {
            "status": "error",
            "message": "TimeoutError: The request timed out."
        }
    except Exception as e:
        return {
            "status": "error",
            "message": f"Unexpected error occurred: {str(e)}"
        }

In [21]:
getvideo.run("BV1VC7g6vE9f")

{'bvid': 'BV1VC7g6vE9f',
 'title': 'Vibe Coding是什么？从AI模型、Agent到工作流，彻底搞懂AI编程工具',
 'desc': '作者知识星球：https://t.zsxq.com/ubYr8\n作者的第一个VibeCoding：https://github.com/cradiator/memory_map_visualizer',
 'owner': {'mid': 16433002,
  'name': '隔壁的程序员老王',
  'face': 'https://i1.hdslb.com/bfs/face/b50c413d89c2596347c3cd5b01af9bf028e1ac40.jpg'},
 'stat': {'aid': 116796937997854,
  'view': 104880,
  'danmaku': 134,
  'reply': 317,
  'favorite': 7695,
  'coin': 2147,
  'share': 771,
  'now_rank': 0,
  'his_rank': 0,
  'like': 4683,
  'dislike': 0,
  'evaluation': '',
  'vt': 0}}

In [23]:
@tool
def getcomment(aid: str, sort: int=3, ps: int=5) -> str:
    '''
    Fetch and summarize comments from a certain video
    Args: aid(str) and parameters
    Returns: str: Comment summary
    '''
    url="https://uapis.cn/api/v1/social/bilibili/replies"
    try:
        response=requests.get(url, params={"oid": aid, "sort": sort}, timeout=5)
        response.raise_for_status()
        data=response.json()

        page = data.get("page", {})
        total_count = page.get("count", 0)
        replies = data.get("replies", [])
        
        if not replies:
            return f"Video (aid: {aid}) has no comments."
            
        comment_lines = [f"Total Video Comments Count: {total_count}\n--- Comments List ---"]
        
        for index, reply in enumerate(replies, 1):
            username = reply.get("member", {}).get("uname", "Unknown")
            message = reply.get("content", {}).get("message", "")
            likes = reply.get("like", 0)
            sub_reply_count = reply.get("count", 0)
            
            if message:
                comment_lines.append(
                    f"[{index}] User: {username} | Likes: {likes} | Sub-replies: {sub_reply_count}\n"
                    f"Content: {message}"
                )
        
        result = "\n\n".join(comment_lines)
        return result

    except requests.exceptions.Timeout:
        return "TimeoutError"
    except Exception as e:
        return f"error {e}"
        

In [25]:
getcomment.run("116796937997854", sort=3)

'Total Video Comments Count: 317\n--- Comments List ---\n\n[1] User: geniusjees | Likes: 74 | Sub-replies: 6\nContent: Vide coding 编码一时爽，debug 火葬场。\n\n[2] User: Nightchrysan | Likes: 71 | Sub-replies: 4\nContent: 真正聪明程序员是自己搭架构，搭业务逻辑，然后让ai写一些核心方法，优化查询语句，或者为代码优化提出一些意见，而不是当甩手掌柜。外行只会指挥黑盒让ai干这干那，然后变着法子的优化自己的嘴皮子，这就像公司研发部门来了一个不懂技术的领导一样，只会下指标，提需求，完全不懂技术，不懂客观规律，现实基层技术牛马中人会拒绝你，让你知道不能干，ai可不会，ai那是真的只会睁眼说瞎话忽悠你，遇到一些简单的crud，你觉得你可以用ai抢走程序员的饭碗，但是，等你涉及到底层或者一些复杂业务场景的时候，你只会干瞪眼浪费token。[吃瓜]\n\n[3] User: 我打劫熊熊养你 | Likes: 174 | Sub-replies: 43\nContent: 我是个一点都不会编程的门外汉。用opencode写了一个行业应用软件，写差不多能用了用claude找bug。找了不少问题，还是有bug。昨天今天用codex测试好了，稳定性不错，然后想加入模块，结果codex直接放飞自我，全废了。两天的功夫又要重来'

### Langchain 
$$\text{Output} = \text{Input} \cup \{\text{New\_Key}: \text{Sub\_Chain}(\text{Input})\}$$

In [33]:
import json
from langchain_core.runnables import RunnablePassthrough

prompt1=ChatPromptTemplate.from_messages([
    ("system", "You are a helpful video analyzer that can extract video's aid from user's input"),
    ("human", "{user_input}")
])

model1 = model.bind_tools([getvideo], tool_choice="getvideo")
extract_aid_chain = prompt1 | model1

def run_tools_sequentially(ai_message) -> str:
    try:
        tool1_args = ai_message.tool_calls[0]["args"]
        video_info = getvideo.invoke(tool1_args)
        
        if video_info.get("status") == "error":
            return f"Error: {video_info.get('message')}"
        target_aid = video_info["stat"].get("aid")
        comments_data = getcomment.invoke({"aid": str(target_aid), "sort": 3, "ps": 5})
        return json.dumps(comments_data, ensure_ascii=False)
        
    except Exception as e:
        return f"Error: {e}"

sequential_tools_node = RunnableLambda(run_tools_sequentially)

prompt2 = ChatPromptTemplate.from_messages([
    ("system", "You are a professional text analyst. Below are the real data from the comment section of a certain video. Please create a detailed content outline for these comments, summarizing the core focus of audience discussions, positive public sentiment, negative feedback, and interesting memes."),
    ("human", "Original query: {user_input}\n\nThe comments are:\n{comments_json}")
])

final_chain = (
    RunnablePassthrough.assign(comments_json= extract_aid_chain | sequential_tools_node)
    | prompt2| model
)

In [34]:
response = final_chain.invoke({"user_input": "What's about the comments in video BV1VC7g6vE9f"})

print(response.content)

好的，作为一名专业的文本分析师，以下是根据您提供的视频（BV1VC7g6vE9f）评论区数据生成的详细内容大纲。

---

### 视频评论区内容大纲

**核心议题：** 围绕“Vibe Coding”（一种依赖AI辅助的编程方式）的实践、风险与本质展开激烈讨论。观众主要探讨AI编程的适用边界、对程序员技能的影响，以及“甩手掌柜”式使用AI的潜在危害。

---

#### 一、 正面/中立讨论：AI编程的实践与价值

1.  **AI编程的“爽”与“痛”**
    *   **核心观点：** 编码一时爽，调试火葬场。
    *   **来源：** 评论[1] (geniusjees)
    *   **分析：** 用幽默的对比（编码 vs. debug）精准概括了AI辅助编程的典型体验——快速生成代码容易，但后续排查和修复AI产生的错误却极其耗时。

2.  **AI编程的“正确打开方式”**
    *   **核心观点：** 聪明程序员将AI视为“高级工具”，而非“甩手掌柜”。
    *   **来源：** 评论[3] (Nightchrysan)
    *   **具体应用场景：**
        *   让AI编写核心方法。
        *   让AI优化查询语句。
        *   让AI为代码优化提出意见。
    *   **核心逻辑：** 程序员负责架构设计、业务逻辑等顶层工作，AI负责执行层和优化建议。这是一种“人机协作”而非“完全替代”的模式。

3.  **AI编程的入门门槛**
    *   **核心观点：** Agent（智能体）开发并不难。
    *   **来源：** 评论[2] (第6人)
    *   **具体内容：** 分享了一个从零开始的Agent实践教程链接，暗示AI编程（特别是Agent开发）是可以学习和掌握的。

---

#### 二、 负面/批判性反馈：对“Vibe Coding”的深度反思

1.  **对“外行指挥AI”的强烈批判**
    *   **核心观点：** 外行使用AI编程，如同不懂技术的领导瞎指挥。
    *   **来源：** 评论[3] (Nightchrysan)
    *   **具体比喻：** 将“只会优化嘴皮子”的AI使用者比作“不懂技术的领导”，只